# Question 18 - 12. Integer to Roman

Seven different symbols represent Roman numerals with the following values:

| Symbol | Value |
|---|---|
| I | 1 |
| V | 5 |
| X | 10 |
| L | 50 |
| C | 100 |
| D | 500 |
| M | 1000 |

Roman numerals are formed by appending the conversions of decimal place values from highest to lowest. Converting a decimal place value into a Roman numeral has the following rules:

- If the value does not start with 4 or 9, select the symbol of the maximal value that can be subtracted from the input, append that symbol to the result, subtract its value, and convert the remainder to a Roman numeral.
- If the value starts with 4 or 9 use the **subtractive form** representing one symbol subtracted from the following symbol, for example, 4 is 1 (`I`) less than 5 (`V`): `IV` and 9 is 1 (`I`) less than 10 (`X`): `IX`. Only the following subtractive forms are used: 4 (`IV`), 9 (`IX`), 40 (`XL`), 90 (`XC`), 400 (`CD`) and 900 (`CM`).
- Only powers of 10 (`I`, `X`, `C`, `M`) can be appended consecutively at most 3 times to represent multiples of 10. You cannot append 5 (`V`), 50 (`L`), or 500 (`D`) multiple times. If you need to append a symbol 4 times use the **subtractive form**.

Given an integer, convert it to a Roman numeral.

**Example 1:**

    Input: num = 3749
    Output: "MMMDCCXLIX"
    Explanation:
    3000 = MMM as 1000 (M) + 1000 (M) + 1000 (M)
     700 = DCC as 500 (D) + 100 (C) + 100 (C)
      40 = XL as 10 (X) less of 50 (L)
       9 = IX as 1 (I) less of 10 (X)
    Note: 49 is not 1 (I) less of 50 (L) because the conversion is based on decimal places

**Example 2:**

    Input: num = 58
    Output: "LVIII"
    Explanation:
    50 = L
     8 = VIII

**Example 3:**

    Input: num = 1994
    Output: "MCMXCIV"
    Explanation:
    1000 = M
     900 = CM
      90 = XC
       4 = IV

**Constraints:**

- `1 <= num <= 3999`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy 'making change' with an ordered table of 13 values

📘 **Revise first:** [Pattern H · simulate carefully](./0-concepts-array-and-string.ipynb) · [Part 5.5 · building strings with join](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (7 symbols, then patch bad runs) | O(1)* | O(1) |
| 2️⃣ Optimized (13-value greedy) | O(1)* | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Turn a normal number (1 to 3999) into Roman numerals. `58` → `LVIII`, `1994` → `MCMXCIV`.

**Everyday picture: a cashier making change.** Suppose you owe someone 1994 and your "coins" are worth 1000, 500, 100, 50, 10, 5 and 1. A cashier hands over the **biggest coin that still fits**, again and again: 1000, then 500 (fits into 994)… and so on.

But Roman numerals have a rule: you **can't** write `CCCC` for 400 or `VIIII` for 9. You must use the "subtractive" forms `CD` (400) and `IX` (9).

**The neat fix:** treat those special forms as **extra coins**. Add `CM = 900`, `CD = 400`, `XC = 90`, `XL = 40`, `IX = 9` and `IV = 4` to the coin tray. Now there are **13 coins**, and the plain cashier method, "biggest coin that fits", produces exactly the correct Roman numeral every time.

### Step 0 · Clarify before coding

🗣️ *"I convert an integer from 1 to 3999 into a Roman numeral, using the largest symbols first, at most three repeats of I, X, C, M, and subtractive forms for 4s and 9s."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Range? | `1 ≤ num ≤ 3999`. | Output is at most 15 characters, so the work is effectively constant. |
| `49`? | `XLIX` (40 + 9), **not** `IL`. | Subtraction works per decimal place. Our coin list handles this automatically. |
| Output case? | Uppercase string. | |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** make change using only the **7 basic symbols**, which can produce forbidden runs like `IIII` or `DCCCC`. Then **patch** those runs afterwards with text replacements (`DCCCC → CM`, `CCCC → CD`, …).

🗣️ *"A first idea is greedy with the seven basic symbols, then post-process the illegal runs into subtractive forms. It works, but the patches must be applied in the right order: VIIII must become IX before IIII becomes IV, or 9 turns into VIV. That's fragile."*

In [1]:
class SolutionBrute:
    BASIC = [(1000, "M"), (500, "D"), (100, "C"), (50, "L"), (10, "X"), (5, "V"), (1, "I")]
    # order matters: fix the 9-patterns before the 4-patterns
    PATCHES = [("DCCCC", "CM"), ("CCCC", "CD"), ("LXXXX", "XC"), ("XXXX", "XL"), ("VIIII", "IX"), ("IIII", "IV")]

    def intToRoman(self, num: int) -> str:
        out = []
        for value, sym in self.BASIC:
            count, num = divmod(num, value)
            out.append(sym * count)
        s = "".join(out)
        for bad, good in self.PATCHES:
            s = s.replace(bad, good)
        return s

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** instead of making mistakes and patching them, **never make them**. Put the six subtractive forms in the coin list. With all 13 values sorted from biggest to smallest, "take the biggest that fits" gives the correct answer directly.

**Why the greedy is safe here:** Roman numerals are written one decimal place at a time (thousands, hundreds, tens, ones). Within each place, the 13 values can build every digit 0–9 the standard way: 8 = `V`+`III`, 9 = `IX`, 4 = `IV`. Greedy picks exactly those.

**What data structure?** A **list of (value, symbol) pairs in descending order**. A dict would work for lookups, but here we need a guaranteed **order**, and a list states that plainly.

**How to build the output string:** collect pieces in a list and `"".join(...)` once at the end (concepts notebook, Part 5.5).

🗣️ *"I'll treat it like making change. I list all 13 values, including the six subtractive pairs like 900 for CM and 4 for IV, from largest to smallest. For each value, I take as many copies as fit, append the symbol that many times, and subtract. Because the subtractive forms are in the list, I never produce illegal runs like IIII, so there's no patching. The input is at most 3999, so the work is bounded: O(1)."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| 7 symbols + patch replacements | ⚠️ | Correct only if patches run in the right order. Easy to break. |
| **13 ordered (value, symbol) pairs + greedy** | ✅ **best** | No special cases. The list order *is* the algorithm. |
| Per-digit lookup tables (`thousands[n//1000] + hundreds[...] + ...`) | ✅ alternative | Also O(1) and branch-free, but you must hand-type 4 tables of 10 strings. Typos are likely under pressure. |
| Recursion | ❌ | No benefit over a simple loop. |

In [2]:
class Solution:
    VALUES = [
        (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
        (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
        (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
    ]

    def intToRoman(self, num: int) -> str:
        out = []
        for value, sym in self.VALUES:
            if num == 0:
                break
            count, num = divmod(num, value)   # how many of this "coin" fit, and what's left
            out.append(sym * count)
        return "".join(out)

### Step 3 · Toy example walkthrough (tell it like a story)

`num = 1994`

| coin value | symbol | how many fit | written so far | left to pay |
|---|---|---|---|---|
| 1000 | M | 1 | M | 994 |
| 900 | CM | 1 | MCM | 94 |
| 500, 400, 100 | n/a | 0 | MCM | 94 |
| 90 | XC | 1 | MCMXC | 4 |
| 50, 40, 10, 9, 5 | n/a | 0 | MCMXC | 4 |
| 4 | IV | 1 | **MCMXCIV** | 0 → stop |

**Narrated:** "1994: one thousand fits, so M, leaving 994. Nine hundred fits, so CM, leaving 94. Five hundred, four hundred and one hundred don't fit. Ninety does, so XC, leaving 4. Then IV. MCMXCIV."

`num = 3749` → MMM (3000) → D (500) → CC (200) → XL (40) → IX (9) = **`MMMDCCXLIX`** ✅

In [3]:
cases = [
    (3749, "MMMDCCXLIX"), (58, "LVIII"), (1994, "MCMXCIV"),
    (1, "I"), (4, "IV"), (9, "IX"), (49, "XLIX"), (3999, "MMMCMXCIX"),
]
for num, expected in cases:
    assert SolutionBrute().intToRoman(num) == expected
    assert Solution().intToRoman(num) == expected
    print(num, "->", expected)

# round-trip every value 1..3999 with a decoder
def from_roman(s):
    v = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    return sum(-v[c] if i + 1 < len(s) and v[c] < v[s[i + 1]] else v[c] for i, c in enumerate(s))

for n in range(1, 4000):
    r = Solution().intToRoman(n)
    assert from_roman(r) == n and r == SolutionBrute().intToRoman(n)
print("All tests passed ✅ (including every value 1..3999)")

3749 -> MMMDCCXLIX
58 -> LVIII
1994 -> MCMXCIV
1 -> I
4 -> IV
9 -> IX
49 -> XLIX
3999 -> MMMCMXCIX
All tests passed ✅ (including every value 1..3999)


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| 7 symbols + patch | O(1)* | O(1)* | Bounded output, but extra string passes. |
| **13-value greedy** | **O(1)*** | **O(1)*** | Always 13 loop steps; output ≤ 15 characters. |

\* **Why "O(1)"?** The loop always runs over the same 13 coins, and because `num ≤ 3999` the output is at most 15 characters (`MMMDCCCLXXXVIII`). The work can't grow beyond a fixed limit. (If numbers could be unlimited, the leading `M`s would make it O(num / 1000).)

**Edge cases to mention:** every 4 and 9 (4, 9, 40, 90, 400, 900), 49 → `XLIX` (not `IL`), 3999.

**Likely follow-ups:**
- *"Why does greedy work here when it fails for some coin systems?"* → For coins {1, 3, 4} and amount 6, greedy gives 4 + 1 + 1 (3 coins) but 3 + 3 (2 coins) is better. Roman values work out per decimal place, so the biggest-first choice is always the standard form.
- *The reverse direction?* → Roman to Integer (Q17).

---

#### 🗣️ Full interview script (about 45 seconds)

*"I'll treat this like making change. I put all thirteen values in descending order, including the six subtractive forms like 900 for CM and 4 for IV.*

*For each value, I take as many as fit, append the symbol that many times, and subtract from the number. Including the subtractive forms as their own coins means I never produce illegal runs like IIII, so no patching is needed.*

*The loop is over a fixed 13 entries, and the input is capped at 3999, so the output is at most 15 characters. Time and space are O(1)."*